# Gating threshold study: analysis

This notebook finds the best threshold for each welfare gate on the **Balanced hybrid** recommender, after the threshold frontiers
of Uğurel & Yabe (2026, Figs 2–3). It reads only the result files that `run_experiments.ipynb` writes to `results/`. It needs nothing beyond pandas, numpy and matplotlib,
so it runs on any machine that has a copy of that folder.

* **PUP** passes a venue only if $\Pr[U \ge 0] \ge \alpha$.
* **RM** passes a venue only if $\mathbb{E}[\text{regret}] \le \varepsilon$.

When every candidate is withheld, the RS *abstains* and the agent goes wherever it would have gone organically.

**Metric.** As in the paper (Table 1): the mean net utility $U = V - C$ of leisure trips on the last (evaluation) day. Each (city, seed)
run gives one mean. Cities are averaged with equal weight, so each seed yields one number. Error bars are the standard deviation of that
number across seeds ($\sigma_U$). The optimal threshold is the one with the highest mean net utility.

Abstention is a fraction of recommendation opportunities, not a utility, so it gets its own panel on the same threshold axis instead
of sharing the utility scale.

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

RESULTS_DIR = Path("results")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

if not (RESULTS_DIR / "DONE.txt").exists():
    raise FileNotFoundError("results/DONE.txt is missing: the experiments have not finished, or the folder was copied incompletely.")

config = json.loads((RESULTS_DIR / "config.json").read_text())
RM_EPSILONS = {int(p): eps for p, eps in
               json.loads((RESULTS_DIR / "rm_epsilons.json").read_text())["epsilon_by_percentile"].items()}
runs = pd.concat([pd.read_csv(RESULTS_DIR / f"{city}.csv") for city in config["cities"]], ignore_index=True)

print(f"{len(config['cities'])} cities x {len(config['seeds'])} seeds, {config['n_agents']} agents, "
      f"{config['n_days']} days: {len(runs)} runs")

# Figure style (reference data-viz palette)
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.axisbelow": True,
})

In [ ]:
ARMS = runs[["arm", "gate", "alpha", "percentile"]].drop_duplicates("arm")


def arm_key(gate, alpha=None, percentile=None):
    # The key of the one arm with this gate and threshold.
    match = ARMS["gate"] == gate
    if alpha is not None:
        match &= np.isclose(ARMS["alpha"], alpha)
    if percentile is not None:
        match &= ARMS["percentile"] == percentile
    (key,) = ARMS.loc[match, "arm"]
    return key


def seed_stats(column):
    # Mean of `column` over cities (equal weight) per (arm, seed), then mean and std over seeds.
    per_seed = runs.groupby(["arm", "seed"])[column].mean()
    return per_seed.groupby(level="arm").agg(["mean", lambda x: x.std(ddof=0)]).set_axis(["mean", "std"], axis=1)


utility, abstention = seed_stats("mean_utility"), seed_stats("abstention_rate")
CONTROLS = {"No RS": arm_key("no_rs"), "Standard RS": arm_key("standard"), "Oracle": arm_key("oracle")}
utility.loc[list(CONTROLS.values())].set_axis(list(CONTROLS), axis=0).round(4)

In [ ]:
def plot_frontier(x, keys, xlabel, best_index, tick_labels=None):
    # Threshold frontier: mean net utility with the controls as flat lines (top), abstention rate (bottom).
    fig, (ax_u, ax_a) = plt.subplots(2, 1, sharex=True, figsize=(10.5, 6.8), height_ratios=[3, 2])
    u, a = utility.loc[keys], abstention.loc[keys]

    for (name, key), color in zip(CONTROLS.items(), [GRAY, ORANGE, AQUA]):
        value = utility.loc[key, "mean"]
        ax_u.axhline(value, color=color, linestyle="--", linewidth=1.5, label=f"{name} ({value:.3f})")
    ax_u.errorbar(x, u["mean"], yerr=u["std"], color=BLUE, marker="o", markersize=6, linewidth=2,
                  capsize=3, label="Gated Balanced hybrid")
    ax_u.plot(x[best_index], u["mean"].iloc[best_index], "o", markersize=15, markerfacecolor="none",
              markeredgecolor=BLUE, markeredgewidth=1.5, label="Optimum")
    ax_u.set_ylabel("Mean net utility\nper leisure trip")
    ax_u.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=False)

    ax_a.errorbar(x, a["mean"], yerr=a["std"], color=BLUE, marker="s", markersize=6, linewidth=2, capsize=3)
    ax_a.set_ylim(0, 1.05)
    ax_a.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax_a.set_ylabel("Abstention rate")
    ax_a.set_xticks(x, tick_labels if tick_labels else [f"{xi:g}" for xi in x])
    ax_a.set_xlabel(xlabel)
    fig.align_ylabels()
    fig.tight_layout()
    return fig


def frontier_table(keys, **threshold_columns):
    return pd.DataFrame({
        **threshold_columns,
        "mean_utility": utility.loc[keys, "mean"].values,
        "sigma_u": utility.loc[keys, "std"].values,
        "abstention": abstention.loc[keys, "mean"].values,
    }).round(4)

## PUP threshold frontier (paper Fig. 2)

Raising $\alpha$ withholds recommendations that are less likely to leave the agent better off. At $\alpha = 0$ nothing is withheld, which is
Standard RS (the orange line).

In [ ]:
alphas = config["pup_alphas"]
pup_keys = [arm_key("pup", alpha=alpha) for alpha in alphas]
best = int(np.argmax(utility.loc[pup_keys, "mean"]))
ALPHA_STAR = alphas[best]

fig = plot_frontier(alphas, pup_keys, r"PUP threshold $\alpha$", best)
fig.savefig(FIG_DIR / "pup_frontier.png", bbox_inches="tight")
plt.show()

print(f"alpha* = {ALPHA_STAR:.1f}")
frontier_table(pup_keys, alpha=alphas)

## RM threshold frontier (paper Fig. 3)

Each point is one calibrated $\varepsilon$: the given percentile of the expected regret of Standard RS's suggestions. The points are spaced by
percentile, because the $\varepsilon$ values themselves are very skewed. A lower $\varepsilon$ tolerates less expected regret and withholds more.

In [ ]:
percentiles = config["rm_percentiles"]
epsilons = [RM_EPSILONS[p] for p in percentiles]
rm_keys = [arm_key("rm", percentile=p) for p in percentiles]
best = int(np.argmax(utility.loc[rm_keys, "mean"]))
P_STAR, EPSILON_STAR = percentiles[best], epsilons[best]

fig = plot_frontier(percentiles, rm_keys, r"RM regret ceiling $\varepsilon$ (percentile of Standard RS's expected regret)", best,
                    tick_labels=[f"p{p}\n{eps:.2f}" for p, eps in zip(percentiles, epsilons)])
fig.savefig(FIG_DIR / "rm_frontier.png", bbox_inches="tight")
plt.show()

print(f"epsilon* = {EPSILON_STAR:.4f} (p{P_STAR})")
frontier_table(rm_keys, percentile=percentiles, epsilon=epsilons)

## Optimal thresholds

The best threshold for each gate, with its gain over Standard RS. That gain is computed per seed and then averaged, so its spread across
seeds shows whether the gate reliably beats the ungated recommender.

In [ ]:
per_seed = runs.groupby(["arm", "seed"])["mean_utility"].mean()
standard = per_seed.loc[CONTROLS["Standard RS"]]
optima = {
    "PUP": (f"alpha = {ALPHA_STAR:.1f}", arm_key("pup", alpha=ALPHA_STAR)),
    "RM": (f"epsilon = {EPSILON_STAR:.4f} (p{P_STAR})", arm_key("rm", percentile=P_STAR)),
}
pd.DataFrame({
    gate: {
        "threshold": threshold,
        "mean_utility": round(utility.loc[key, "mean"], 4),
        "abstention": round(abstention.loc[key, "mean"], 4),
        "gain_vs_standard": round((per_seed.loc[key] - standard).mean(), 4),
        "gain_sd_over_seeds": round((per_seed.loc[key] - standard).std(ddof=0), 4),
        "seeds_beating_standard": f"{int((per_seed.loc[key] > standard).sum())} of {len(standard)}",
    }
    for gate, (threshold, key) in optima.items()
}).T